# 10. Artificial Recharge Assessment

This notebook covers the second project component: artificial recharge potential assessment.

The groundwater prediction model is already completed and saved in `models/best_model.pkl`.
No model training is done here.

Because reliable station-level recharge labels are not available, this notebook uses a transparent rule-based scoring method for decision support.


## Introduction

Groundwater recharge potential is estimated from observed groundwater behaviour at each station.
The approach is intentionally simple and explainable so the assumptions can be defended during project review.


## Objectives

- Build a station-wise recharge potential score without supervised machine learning.
- Classify stations into Low, Medium, and High recharge potential.
- Visualize spatial and statistical recharge patterns.
- Document assumptions, limitations, and interpretation boundaries.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")


## Dataset Used

Input dataset:
`data/processed/groundwater_feature_engineered.csv`

This is the processed and feature-engineered dataset from previous notebooks.


In [ ]:
candidate_paths = [
    Path.cwd().resolve() / "data" / "processed" / "groundwater_feature_engineered.csv",
    Path.cwd().resolve().parent / "data" / "processed" / "groundwater_feature_engineered.csv",
    Path("../data/processed/groundwater_feature_engineered.csv"),
    Path("data/processed/groundwater_feature_engineered.csv")
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("groundwater_feature_engineered.csv not found in expected paths.")

df = pd.read_csv(dataset_path)
df["Data Acquisition Time"] = pd.to_datetime(df["Data Acquisition Time"], errors="coerce")

df.head()


## Methodology

A rule-based station score is built from available variables only:

- Average groundwater depth (`Groundwater Level Telemetry 6 Hourly (meter)`)  
- Depth fluctuation (`std`, range, and `RollingStd_4`)  
- Long-term depth trend over time  
- RL_MSL as a weak terrain-related proxy  
- Spatial location for mapping and regional comparison

No artificial recharge labels are created, and no supervised model is used.


## Assumptions

1. Higher average groundwater depth indicates larger recharge need and potential benefit.
2. Higher fluctuation indicates stronger response of groundwater system and possible recharge sensitivity.
3. A long-term increasing depth trend indicates depletion pressure, so recharge potential/priority is higher.
4. RL_MSL is used only as a low-weight proxy because hydrogeology, soil, and infiltration tests are unavailable.
5. Rainfall, soil texture, land use, geology, and aquifer transmissivity are not included because those fields are not present in the dataset.


## Recharge Scoring Method

Station score components (0 to 100 after scaling):

- **Depth Score (40%)**: normalized station average groundwater depth
- **Fluctuation Score (30%)**: normalized combination of depth variability metrics
- **Trend Score (20%)**: normalized positive depth trend (depletion-oriented)
- **RL_MSL Proxy Score (10%)**: normalized RL_MSL

Final score = weighted sum of the four components.

Categories are assigned using score tertiles:
- Low Recharge Potential
- Medium Recharge Potential
- High Recharge Potential


In [ ]:
target_col = "Groundwater Level Telemetry 6 Hourly (meter)"

analysis_df = df[[
    "Station", "Latitude", "Longitude", "Data Acquisition Time",
    target_col, "RL_MSL", "RollingStd_4", "RollingMean_4"
]].copy()

analysis_df = analysis_df.dropna(subset=["Station", target_col, "Latitude", "Longitude", "Data Acquisition Time"])
analysis_df = analysis_df.sort_values(["Station", "Data Acquisition Time"]).reset_index(drop=True)

station_summary = (
    analysis_df.groupby("Station", as_index=False)
    .agg(
        Latitude=("Latitude", "first"),
        Longitude=("Longitude", "first"),
        Avg_Depth=(target_col, "mean"),
        Min_Depth=(target_col, "min"),
        Max_Depth=(target_col, "max"),
        Depth_Std=(target_col, "std"),
        Avg_RollingStd4=("RollingStd_4", "mean"),
        Avg_RL_MSL=("RL_MSL", "mean"),
        Observations=(target_col, "count")
    )
)

station_summary["Depth_Range"] = station_summary["Max_Depth"] - station_summary["Min_Depth"]
station_summary["Depth_Std"] = station_summary["Depth_Std"].fillna(0)
station_summary["Avg_RollingStd4"] = station_summary["Avg_RollingStd4"].fillna(station_summary["Avg_RollingStd4"].median())
station_summary["Avg_RL_MSL"] = station_summary["Avg_RL_MSL"].fillna(station_summary["Avg_RL_MSL"].median())

station_summary.head()


In [ ]:
def minmax_scale(series):
    min_val = series.min()
    max_val = series.max()
    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return pd.Series(np.full(len(series), 50.0), index=series.index)
    return ((series - min_val) / (max_val - min_val)) * 100

trend_rows = []
for station, part in analysis_df.groupby("Station"):
    part = part.sort_values("Data Acquisition Time")
    x = np.arange(len(part))
    y = part[target_col].values

    if len(part) < 2 or np.all(y == y[0]):
        slope = 0.0
    else:
        slope = np.polyfit(x, y, 1)[0]

    trend_rows.append({"Station": station, "Depth_Trend_Slope": slope})

trend_df = pd.DataFrame(trend_rows)
station_summary = station_summary.merge(trend_df, on="Station", how="left")

station_summary["Depth_Score"] = minmax_scale(station_summary["Avg_Depth"])

station_summary["Fluctuation_Combined"] = (
    0.50 * minmax_scale(station_summary["Depth_Std"]) +
    0.30 * minmax_scale(station_summary["Depth_Range"]) +
    0.20 * minmax_scale(station_summary["Avg_RollingStd4"])
)

positive_trend = station_summary["Depth_Trend_Slope"].clip(lower=0)
station_summary["Trend_Score"] = minmax_scale(positive_trend)
station_summary["RL_MSL_Score"] = minmax_scale(station_summary["Avg_RL_MSL"])

station_summary["Recharge_Score"] = (
    0.40 * station_summary["Depth_Score"] +
    0.30 * station_summary["Fluctuation_Combined"] +
    0.20 * station_summary["Trend_Score"] +
    0.10 * station_summary["RL_MSL_Score"]
)

q1 = station_summary["Recharge_Score"].quantile(1/3)
q2 = station_summary["Recharge_Score"].quantile(2/3)

station_summary["Recharge_Category"] = pd.cut(
    station_summary["Recharge_Score"],
    bins=[-np.inf, q1, q2, np.inf],
    labels=["Low Recharge Potential", "Medium Recharge Potential", "High Recharge Potential"],
    include_lowest=True
)

station_summary = station_summary.sort_values("Recharge_Score", ascending=False).reset_index(drop=True)


## Analysis

### Recharge Score Table


In [ ]:
recharge_score_table = station_summary[[
    "Station", "Latitude", "Longitude", "Observations",
    "Avg_Depth", "Depth_Std", "Depth_Range", "Depth_Trend_Slope",
    "Recharge_Score", "Recharge_Category"
]].copy()

recharge_score_table


### Recharge Category Table


In [ ]:
recharge_category_table = (
    station_summary["Recharge_Category"]
    .value_counts()
    .rename_axis("Recharge_Category")
    .reset_index(name="Station_Count")
)

recharge_category_table


### Summary Statistics


In [ ]:
summary_stats = station_summary[[
    "Recharge_Score", "Avg_Depth", "Depth_Std", "Depth_Range", "Depth_Trend_Slope"
]].describe().T

summary_stats


### Top and Bottom Stations


In [ ]:
print("Top 5 stations with highest recharge potential:")
display(station_summary[["Station", "Recharge_Score", "Recharge_Category"]].head(5))

print("Bottom 5 stations with lowest recharge potential:")
display(station_summary[["Station", "Recharge_Score", "Recharge_Category"]].tail(5))


## Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(station_summary["Recharge_Score"], bins=8, kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Distribution of Recharge Scores")
axes[0].set_xlabel("Recharge Score")

sns.countplot(
    data=station_summary,
    x="Recharge_Category",
    order=["Low Recharge Potential", "Medium Recharge Potential", "High Recharge Potential"],
    palette="viridis",
    ax=axes[1]
)
axes[1].set_title("Recharge Category Distribution")
axes[1].set_xlabel("Recharge Category")
axes[1].set_ylabel("Number of Stations")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    station_summary["Longitude"],
    station_summary["Latitude"],
    c=station_summary["Recharge_Score"],
    s=station_summary["Recharge_Score"] * 4 + 60,
    cmap="YlGnBu",
    edgecolor="black",
    linewidth=0.4,
    alpha=0.9
)

for _, row in station_summary.iterrows():
    plt.text(row["Longitude"] + 0.004, row["Latitude"] + 0.004, row["Station"], fontsize=8)

plt.colorbar(scatter, label="Recharge Score")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Spatial Recharge Potential Map")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
lat_median = station_summary["Latitude"].median()
lon_median = station_summary["Longitude"].median()

station_summary["Lat_Band"] = np.where(station_summary["Latitude"] >= lat_median, "North", "South")
station_summary["Lon_Band"] = np.where(station_summary["Longitude"] >= lon_median, "East", "West")
station_summary["Region"] = station_summary["Lat_Band"] + "-" + station_summary["Lon_Band"]

regional_comparison = (
    station_summary.groupby("Region", as_index=False)
    .agg(
        Avg_Recharge_Score=("Recharge_Score", "mean"),
        Mean_Depth=("Avg_Depth", "mean"),
        Station_Count=("Station", "count")
    )
    .sort_values("Avg_Recharge_Score", ascending=False)
)

regional_comparison


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=regional_comparison, x="Region", y="Avg_Recharge_Score", palette="mako")
plt.title("Regional Comparison of Average Recharge Score")
plt.xlabel("Region")
plt.ylabel("Average Recharge Score")
plt.tight_layout()
plt.show()


## Observations

- Stations with higher average groundwater depth and stronger variability generally received higher recharge scores.
- Stations with low depth variation and stable trend generally fell in the low recharge category.
- Spatial clustering is visible, indicating that recharge behaviour is not uniform across the study area.
- Regional grouping shows that some quadrants have systematically higher recharge potential scores.


## Discussion

Recharge potential differs between stations because groundwater behaviour differs in depth, fluctuation, and trend.
Deeper and more variable groundwater conditions often indicate stronger need and scope for recharge interventions.

This is an estimation framework, not ground truth, because validated recharge labels are unavailable.
The score is therefore best used for station prioritization and planning support.


## Limitations

- No measured recharge labels are available for supervised calibration.
- Rainfall, soil type, land use, geology, and pumping stress are not available in the current dataset.
- RL_MSL is only a proxy and cannot replace detailed hydrogeological field information.
- Station-level aggregation can hide seasonal and short-term local effects.


## Conclusion

This notebook complements the completed groundwater prediction model by providing a scientifically justified artificial recharge assessment using available groundwater observations.

Because recharge labels are unavailable, this output is a **decision-support assessment** based on transparent rules, not a supervised recharge prediction model.
